In [0]:
%pip install catboost lightgbm optuna holidays duckdb

In [0]:
import numpy as np
import pandas as pd
import duckdb

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

import holidays

In [0]:
def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

DRIVE_FOLDER_NAME = "Store Sales"

In [0]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [0]:
TARGET = "sales"
N_SPLITS = 5
SEED = 42
eps = 1e-5

def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

non_relevant_columns = ['id', 'source']


In [0]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/train.csv')
    test_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  train_df['date'] =  pd.to_datetime(train_df['date'])
  test_df['date'] =  pd.to_datetime(test_df['date'])
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

In [0]:
def load_helper_data(option: str="local"):
  '''
  returns oil_df, transactions_df, holidays_events_df, stores_df

  '''
  if option == "local":
    oil_df = pd.read_csv('oil.csv')
    transactions_df = pd.read_csv('transactions.csv')
    holidays_events_df = pd.read_csv('holidays_events.csv')
    stores_df = pd.read_csv('stores.csv')


  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    oil_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/oil.csv')
    transactions_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/transactions.csv')
    holidays_events_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/holidays_events.csv')
    stores_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/stores.csv')
    
  oil_df['date'] =  pd.to_datetime(oil_df['date'])
  transactions_df['date'] =  pd.to_datetime(transactions_df['date'])
  holidays_events_df['date'] =  pd.to_datetime(holidays_events_df['date'])
  


  return oil_df, transactions_df, holidays_events_df, stores_df

In [0]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

In [0]:
train_df, test_df, df = load_data(ENVIRONMENT)
oil_df, transactions_df, holidays_events_df, stores_df = load_helper_data(ENVIRONMENT)

# EDA

## Overall Info

### Main DF

In [0]:
df.head()

In [0]:
df.groupby('source').agg(min_date=('date', 'min'), max_date=('date', 'max'))

In [0]:
df.describe(include='all')

### Stores

In [0]:
stores_df.head()

In [0]:
stores_df.describe(include='all')

### Transactions

In [0]:
transactions_df.head()

In [0]:
transactions_df.describe(include='all')

### Holidays

In [0]:
holidays_events_df.head()

In [0]:
holidays_events_df.describe()

### Oil

In [0]:
oil_df.head()

In [0]:
oil_df.describe()

In [0]:
df.head()

In [0]:
transactions_df

# Merging datasets

In [0]:
initial_length = len(df)

In [0]:
df.isna().sum()

In [0]:
temp_df = train_df.merge(transactions_df, on=['date', 'store_nbr'], how='left')

In [0]:
duckdb.sql("""--sql
    SELECT
        SUM(CASE WHEN sales = 0 THEN 1 ELSE 0 END) AS zero_sales,
        SUM(CASE WHEN sales > 0 THEN 1 ELSE 0 END) AS more_sales
    FROM temp_df
    WHERE transactions IS NULL
""").df()

In [0]:
temp_df.loc[(temp_df['sales'] > 0) & (temp_df['transactions'].notna())]

In [0]:
mask = temp_df["transactions"].notna()

ratio_table = temp_df.loc[mask].groupby(["store_nbr", "family"]).agg(
    total_sales=("sales", "sum"),
    total_transactions=("transactions", 'sum')
).reset_index()

ratio_table['ratio'] = ratio_table['total_sales'] / ratio_table['total_transactions']
display(ratio_table)

In [0]:
is_transferred = holidays_events_df['transferred'] == True
is_national = holidays_events_df['locale'] == 'National'
is_regional = holidays_events_df['locale'] == 'Regional'
is_local = holidays_events_df['locale'] == 'Local'

national_holidays = holidays_events_df.loc[(~is_transferred) & (is_national)]
regional_holidays = holidays_events_df.loc[(~is_transferred) & (is_regional)]
local_holidays = holidays_events_df.loc[(~is_transferred) & (is_local)]

In [0]:
holidays_events_df[holidays_events_df["locale"] == "National"][["date"]].drop_duplicates().assign(is_national=1)

In [0]:
duplicated = national_holidays.loc[national_holidays['date'].duplicated()]['date']
national_holidays.loc[national_holidays['date'].isin(duplicated)]

In [0]:
df

In [0]:
temp_df = (df.merge(transactions_df, on=['date', 'store_nbr'], how='left')
            .merge(stores_df, on='store_nbr', how='left')
            .merge(oil_df, on='date', how='left')
            .merge(national_holidays, on='date', how='left')
            .merge(regional_holidays, left_on=['date', 'state'], right_on=['date','locale_name'], how='left')
            # .merge(local_holidays, left_on=['date','city'], right_on='locale_name', how='left')
            )

temp_df

In [0]:
assert len(df) == len(transactions_df), 'Not equal'

In [0]:
stores_df['city'].unique()

In [0]:
display(stores_df)

In [0]:
df.sort_values(by='date', ascending=True).reset_index(drop=True).head(10)

In [0]:
df.index[df['date'] == '2017-01-01']

In [0]:
dates = np.sort(df["date"].unique())

tscv = TimeSeriesSplit(n_splits=4, test_size=16)   # 16 days = competition horizon

for i in enumerate(tscv.split(dates)):
    print(i)

In [0]:
stores_df['state'].unique()